In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
import torch
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)


In [ ]:
# 2. Create TensorDataset objects
from torch.utils.data import TensorDataset
train_dataset = TensorDataset(X_train , y_train)
test_dataset = TensorDataset(X_test , y_test)





In [ ]:
# 3. Create DataLoaders
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)



In [ ]:
# 4. Print shape of one batch
#as it comes with image, y
print(next(iter(train_loader))[0].shape)


In [ ]:
# 5. Display sample images
import matplotlib.pyplot as plt

images, target = next(iter(train_loader))
plt.figure(figsize=(8, 4))

for i in range(6):
    plt.subplot(2, 3, i + 1)

    img = images[i].permute(1, 2, 0)

    plt.imshow(img)
    plt.title(f"Label: {target[i].item()}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Task 1: Write your model class here:

from torch import nn

class nmodel(nn.Module):
    def __init__(self, num_input, num_hidden, num_output):
        super(nmodel, self).__init__()

        self.layer1 = nn.Linear(num_input, num_hidden)
        self.layer2 = nn.Linear(num_hidden, num_hidden)
        self.layer3 = nn.Linear(num_hidden, num_hidden)
        self.layer4 = nn.Linear(num_hidden, num_output)
        self.activation = nn.ReLU()


    def forward(self, x):
        z1 = self.layer1(x)
        a1 = self.activation(z1)

        z2 = self.layer2(a1)
        a2 = self.activation(z2)

        z3 = self.layer3(a2)
        a3 = self.activation(z3)

        z4 = self.layer4(a3)

        return z4



In [ ]:
# Task 2: Write your training loop here:
def training_loop(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for batch_x, batch_y in train_loader:
        #in images (view)
        batch_x = batch_x.view(batch_x.size(0), -1).to(device)
        batch_y = batch_y.view(-1, 1).to(device)


        preds = model(batch_x)
        loss = criterion(preds, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)



In [ ]:
# Task 3: Write your validation loop here:
def validation_loop(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            #in images
            batch_x = batch_x.view(batch_x.size(0), -1).to(device)
            batch_y = batch_y.view(-1, 1).to(device)


            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)

            total_loss += loss.item()
    avg_loss = total_loss / len(test_loader)

    return avg_loss


In [ ]:
print(next(iter(train_loader))[0].shape)


In [ ]:
# Task 4: Define device, model, loss, optimizer:
from torch.optim import AdamW

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
#trying num of hidden
model = nmodel(num_input= 3 * 36 * 36, num_hidden=int((3 * 36 * 36)/5), num_output= 1).to(device)
criterion = nn.MSELoss() # as it will preduce numbers
lr = 0.001
optimizer = AdamW(model.parameters(), lr)


In [ ]:
# Task 5: Start training for 20 epochs:
num_of_epochs = 20
train_losses = []
test_losses = []
for epoch in range(num_of_epochs):
    train_loss = training_loop( #training loop function
            model, train_loader, optimizer, criterion, device
        )
    test_loss = validation_loop( #val loop function
            model, test_loader, criterion, device
        )

    print(
            f"Epoch {epoch+1} | "
            f"Train Loss: {train_loss:.4f} | "
            f"test Loss: {test_loss:.4f} | "
        )
    train_losses+=[train_loss]
    test_losses+=[test_loss]



In [ ]:
print(train_losses)

In [ ]:
# Task 1: Write your code here:
# Plotting results
plt.figure(figsize=(7, 5))

plt.plot(train_losses, label='Train Loss')
plt.plot(test_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here:

import matplotlib.pyplot as plt
images, target = next(iter(train_loader))

with torch.no_grad():
  predictions = model(batch_x.flatten(start_dim=1).to(device))

plt.figure(figsize=(8, 4))

for i in range(6):
    plt.subplot(2, 3, i + 1)

    img = images[i].permute(1, 2, 0)

    plt.imshow(img)
    plt.title(f"true: {target[i].item()}, pred: {int(predictions[i])}")
    plt.axis('off')

plt.tight_layout()
plt.show()